# 16 — Paper Figures: Classifier and Augmentation Choice

Three panels. (a) and (b) are ranked bar-and-dot comparisons in the
SEP_DataAugmentation-2 style. (c) is new: it answers *why* classical
oversampling (SMOTE/ADASYN) beats generative augmentation (TimeGAN/
Diffusion) rather than just reporting that it does.

The mechanism is not what a first guess would suggest. False-alarm rate
(FAR) is nearly identical across all four algorithms (~0.95-0.96) --
generative methods are not more cautious. The entire gap is **recall**:
classical methods catch ~55% of real SEP events, generative methods
catch only ~36-41%. Panel (c) makes that the headline instead of a
footnote.

**Reads:** `./results/*.txt`. **Requires:** notebook 14 run first.
**Writes:** `./paper/figures/design_choices.pdf` / `.png`.


## 1. Load and Preview

In [3]:
# ══════════════════════════════════════════════════════════════
# Best configuration per classifier (same selection notebook 12 uses:
# highest mean TSS per classifier across the whole pipeline).
# ══════════════════════════════════════════════════════════════

BEST_CONFIG = {
    "GRU": "hybrid_tomek",
    "PatchTST": "tomek_rus8000_adasyn8000",
    "SVM": "minmax_all",
    "InceptionTime": "tomek_nonsep_8000",
}
CLF_ORDER = sorted(BEST_CONFIG, key=lambda c: -load_col(
    [k for k, v in CLASSIFIER_LABELS.items() if v == c][0], BEST_CONFIG[c]).mean())

# Augmentation algorithms at their 500-size variant, pooled across all
# four classifiers (matches notebook 12's cross-classifier comparison).
AUG_ORDER = ["ADASYN", "SMOTE", "TimeGAN", "Diffusion"]
AUG_KEYS  = {"ADASYN": "tomek_rus500_adasyn500", "SMOTE": "tomek_rus500_smote500",
            "TimeGAN": "timegan_500", "Diffusion": "diffusion_500"}
AUG_FAMILY = {"ADASYN": "classical", "SMOTE": "classical",
             "TimeGAN": "generative", "Diffusion": "generative"}
FAMILY_COLOR = {"classical": TEAL, "generative": VERM}

# The four shared RUS & OS configurations, across all four augmentation
# algorithms -- used by panel (c) to sweep TSS the same way notebook 12's
# panel (c) sweeps TSS across RUS severity.
CONFIG_ORDER = ["Balanced", "8000", "2000", "500"]
CONFIG_KEYS = {
    "Balanced": {"ADASYN": "tomek_adasyn_balanced", "SMOTE": "tomek_smote_balanced",
                 "TimeGAN": "timegan_balanced", "Diffusion": "diffusion_balanced"},
    "8000":     {"ADASYN": "tomek_rus8000_adasyn8000", "SMOTE": "tomek_rus8000_smote8000",
                 "TimeGAN": "timegan_8000", "Diffusion": "diffusion_8000"},
    "2000":     {"ADASYN": "tomek_rus2000_adasyn2000", "SMOTE": "tomek_rus2000_smote2000",
                 "TimeGAN": "timegan_2000", "Diffusion": "diffusion_2000"},
    "500":      {"ADASYN": "tomek_rus500_adasyn500", "SMOTE": "tomek_rus500_smote500",
                 "TimeGAN": "timegan_500", "Diffusion": "diffusion_500"},
}


def pooled_config_tss(level):
    """All TSS runs across all 4 classifiers x all 4 algorithms at one
    RUS & OS configuration (32 points: 4 classifiers x 4 algorithms x 2 runs)."""
    out = []
    for key in CONFIG_KEYS[level].values():
        out.extend(load_all_classifiers(key).tolist())
    return np.array(out)


# For panel (c): the single best (algorithm, configuration) combination
# for each classifier, searched only across the 16 over-sampling-experiment
# combinations (4 algorithms x 4 configs) -- not the whole-pipeline best
# BEST_CONFIG above, which can come from a non-augmentation stage
# (e.g. SVM's overall best is Min-Max normalization alone).
BEST_OSRUS = {}
for key_slug, clf in CLASSIFIER_LABELS.items():
    best_mean, best_key, best_label, best_fam = -np.inf, None, None, None
    for level in CONFIG_ORDER:
        for algo, key in CONFIG_KEYS[level].items():
            v = load_col(key_slug, key)
            if v.size and v.mean() > best_mean:
                best_mean, best_key = v.mean(), key
                best_label, best_fam = f"{algo} {level}", AUG_FAMILY[algo]
    BEST_OSRUS[clf] = {"key": best_key, "label": best_label,
                       "family": best_fam, "mean": best_mean}
CLF_OSRUS_ORDER = sorted(BEST_OSRUS, key=lambda c: -BEST_OSRUS[c]["mean"])


print("Best config per classifier:")
for clf in CLF_ORDER:
    key_slug = {v: k for k, v in CLASSIFIER_LABELS.items()}[clf]
    v = load_col(key_slug, BEST_CONFIG[clf])
    print(f"  {clf:<14} {BEST_CONFIG[clf]:<26} mean TSS={v.mean():.3f}")

print("\nAugmentation algorithm comparison (pooled, n=8 each):")
for a in AUG_ORDER:
    v = load_all_classifiers(AUG_KEYS[a])
    r = load_all_classifiers(AUG_KEYS[a], col=COLUMN["recall"])
    f = load_all_classifiers(AUG_KEYS[a], col=COLUMN["far"])
    print(f"  {a:<10} ({AUG_FAMILY[a]:<10}) TSS={v.mean():.3f}  "
          f"Recall={r.mean():.3f}  FAR={f.mean():.3f}")

print("\nTSS pooled across all 4 algorithms x 4 classifiers, per configuration (n=32 each):")
for level in CONFIG_ORDER:
    v = pooled_config_tss(level)
    print(f"  {level:<10} mean TSS={v.mean():.3f}  min={v.min():.3f}  max={v.max():.3f}")

print("\nBest over-sampling-experiment result per classifier (best of 16 algorithm x config combos):")
for clf in CLF_OSRUS_ORDER:
    b = BEST_OSRUS[clf]
    print(f"  {clf:<14} {b['label']:<16} ({b['family']:<10}) mean TSS={b['mean']:.3f}")


Best config per classifier:
  GRU            hybrid_tomek               mean TSS=0.671
  PatchTST       tomek_rus8000_adasyn8000   mean TSS=0.623
  SVM            minmax_all                 mean TSS=0.505
  InceptionTime  tomek_nonsep_8000          mean TSS=0.404

Augmentation algorithm comparison (pooled, n=8 each):
  ADASYN     (classical ) TSS=0.452  Recall=0.555  FAR=0.952
  SMOTE      (classical ) TSS=0.449  Recall=0.559  FAR=0.955
  TimeGAN    (generative) TSS=0.275  Recall=0.360  FAR=0.961
  Diffusion  (generative) TSS=0.323  Recall=0.412  FAR=0.955

TSS pooled across all 4 algorithms x 4 classifiers, per configuration (n=32 each):
  Balanced   mean TSS=0.163  min=-0.006  max=0.613
  8000       mean TSS=0.173  min=-0.008  max=0.654
  2000       mean TSS=0.271  min=-0.006  max=0.669
  500        mean TSS=0.375  min=0.059  max=0.663

Best over-sampling-experiment result per classifier (best of 16 algorithm x config combos):
  GRU            ADASYN 500       (classical ) mean TSS=0

## Figure — Base Learner, Augmentation, and Why It Works

In [4]:
# FIG -- design choices: augmentation, across configurations, and best per classifier
fig, axes = plt.subplots(1, 3, figsize=(7.16, 2.35))

# ---------------------------------------------------------------
# (a) Augmentation algorithm -- classical vs generative, colored by family
# ---------------------------------------------------------------
ax = axes[0]
for i, a in enumerate(AUG_ORDER):
    v = load_all_classifiers(AUG_KEYS[a])
    col = FAMILY_COLOR[AUG_FAMILY[a]]
    ax.bar(i, v.mean(), 0.60, color=col, edgecolor="none", zorder=3)
    jitter = np.linspace(-0.09, 0.09, len(v))
    ax.scatter(np.full_like(v, i, dtype=float) + jitter, v, s=5, color="0.15",
              lw=0, alpha=0.75, zorder=5)
    ax.text(i, v.max() + 0.045, f"{v.mean():.2f}", ha="center", fontsize=5.8,
            color="0.0", zorder=6,
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.82, pad=0.6))
ax.set_xticks(range(len(AUG_ORDER)))
ax.set_xticklabels(AUG_ORDER, rotation=32, ha="right", fontsize=5.9)
for tick, a in zip(ax.get_xticklabels(), AUG_ORDER):
    tick.set_color(FAMILY_COLOR[AUG_FAMILY[a]])
ax.set_ylim(0, 0.78)
finish(ax, ylab="Test TSS")
ax.set_title("(a)  Augmentation", loc="left", fontsize=7.0, pad=4)
ax.text(0.02, 0.99, "classical", transform=ax.transAxes, fontsize=5.8,
        color=TEAL, ha="left", va="top",
        bbox=dict(facecolor="white", edgecolor="none", alpha=0.82, pad=0.6))
ax.text(0.98, 0.99, "generative", transform=ax.transAxes, fontsize=5.8,
        color=VERM, ha="right", va="top",
        bbox=dict(facecolor="white", edgecolor="none", alpha=0.82, pad=0.6))

# ---------------------------------------------------------------
# (b) TSS across RUS & OS configurations -- pooled across all 4
# algorithms and all 4 classifiers at each configuration, in the same
# line + shaded-spread style as notebook 12's panel (c) RUS sweep. The
# best point here always lands on the rightmost ("500") tick, so the
# label is placed above-left of it rather than to the right, where it
# would run off the axes.
# ---------------------------------------------------------------
ax = axes[1]
xs = np.arange(len(CONFIG_ORDER))
means = np.array([pooled_config_tss(level).mean() for level in CONFIG_ORDER])
los   = np.array([pooled_config_tss(level).min() for level in CONFIG_ORDER])
his   = np.array([pooled_config_tss(level).max() for level in CONFIG_ORDER])

ax.fill_between(xs, los, his, color=BLUE, alpha=0.13, lw=0, zorder=2,
                label="4-algorithm spread")
ax.plot(xs, means, "-o", color=BLUE, lw=1.1, ms=3.4, mfc="white", mew=0.9,
        zorder=4, label="mean TSS")
best_i = int(np.argmax(means))
ax.scatter([best_i], [means[best_i]], s=40, facecolor="none", edgecolor=TEAL,
          lw=1.2, zorder=5)
ax.annotate("best on\naverage", xy=(best_i, means[best_i]),
            xytext=(best_i - 0.55, means[best_i] + 0.11), fontsize=5.6,
            color=TEAL, ha="center", linespacing=1.2,
            arrowprops=dict(arrowstyle="-", lw=0.6, color=TEAL))
ax.set_xlim(-0.4, len(CONFIG_ORDER) - 1 + 0.4)
ax.set_xticks(xs)
ax.set_xticklabels(CONFIG_ORDER, fontsize=6.0)
finish(ax, ylab="Test TSS", xlab="RUS & OS configuration")
ax.set_title("(b)  Across configurations", loc="left", fontsize=7.0, pad=4)
ax.legend(frameon=False, loc="upper left", handlelength=1.0, handletextpad=0.4,
          borderpad=0.1, labelspacing=0.25)

# ---------------------------------------------------------------
# (c) Best result per classifier, searched only within this section's
# 16 over-sampling-experiment combinations (4 algorithms x 4 RUS & OS
# configs) -- not the whole-pipeline best, which can come from a
# non-augmentation stage. Bar color marks which family (classical vs
# generative) won for that classifier. If every classifier happens to
# share the same winning algorithm (as ADASYN does here), that's noted
# once above the panel instead of repeated in each bar, so the in-bar
# label can stay short enough to fit even the shortest bar; otherwise
# each bar gets its own "algorithm level" label.
# ---------------------------------------------------------------
ax = axes[2]
winning_algos = {BEST_OSRUS[c]["label"].split()[0] for c in CLF_OSRUS_ORDER}
shared_algo = winning_algos.pop() if len(winning_algos) == 1 else None
for i, clf in enumerate(CLF_OSRUS_ORDER):
    key_slug = {v: k for k, v in CLASSIFIER_LABELS.items()}[clf]
    b = BEST_OSRUS[clf]
    v = load_col(key_slug, b["key"])
    col = FAMILY_COLOR[b["family"]]
    algo, level = b["label"].split()
    in_bar_label = level if shared_algo else b["label"]
    ax.bar(i, v.mean(), 0.60, color=col, edgecolor="none", zorder=3)
    ax.scatter(np.full_like(v, i, dtype=float), v, s=5, color="0.15", lw=0, zorder=5)
    ax.text(i, v.max() + 0.045, f"{v.mean():.2f}", ha="center", fontsize=5.8,
            color="0.0", zorder=6,
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.82, pad=0.6))
    ax.text(i, 0.02, in_bar_label, rotation=90, ha="center", va="bottom",
            fontsize=5.6, color="white", zorder=6)
ax.set_xticks(range(len(CLF_OSRUS_ORDER)))
ax.set_xticklabels(CLF_OSRUS_ORDER, rotation=32, ha="right", fontsize=5.9)
ax.set_ylim(0, 0.78)
finish(ax, ylab="Test TSS")
ax.set_title("(c)  Best per classifier", loc="left", fontsize=7.0, pad=4)
if shared_algo:
    ax.text(0.98, 0.99, f"all {shared_algo}", transform=ax.transAxes, fontsize=5.8,
            color=TEAL, ha="right", va="top",
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.82, pad=0.6))

fig.subplots_adjust(left=0.075, right=0.99, top=0.90, bottom=0.27, wspace=0.42)
fig.savefig(f"{FIG_DIR}/design_choices.pdf")
fig.savefig(f"{FIG_DIR}/design_choices.png", dpi=340)
plt.show()
print("Saved design_choices.pdf/png")


Saved design_choices.pdf/png


/var/folders/fx/gjhbmrbj5jn295_9wrqpbsv80000gn/T/ipykernel_83231/2755240616.py:106: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
